In [275]:
import json
import os

In [276]:
def load_risk_config():
    """Load risk statistics from JSON config"""
    config_path = os.path.join('../../configs/risk_stats.json')
    with open(config_path, 'r') as f:
        return json.load(f)

def load_extraction_config():
    """Load extraction decision rules from JSON config"""
    config_path = os.path.join('../../configs/extraction_type_stats.json')
    with open(config_path, 'r') as f:
        return json.load(f)

def load_binary_config():
    """Load binary removal decision config from JSON"""
    config_path = os.path.join('../../configs/extraction_binary_stats.json')
    with open(config_path, 'r') as f:
        return json.load(f)

In [277]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Helper functions
def generate_binary(n, p):
    return np.random.choice([0, 1], size=n, p=[1-p, p])

def generate_categorical(n, categories, probs):
    return np.random.choice(categories, size=n, p=probs)

def generate_age(n, mu=28, sigma=7, lo=16, hi=60):
    return np.clip(np.random.normal(loc=mu, scale=sigma, size=n).astype(int), lo, hi)

def softmax(vec, temperature=0.8):
    v = np.array(list(vec), dtype=float)
    v = v - v.max()
    ex = np.exp(v / max(1e-6, temperature))
    return ex / ex.sum()

In [278]:
# Non-IID client profiles
CLIENT_PROFILES = {
    1:  {"name": "AcademicCenter",
         "prevalence_shift": {"Impaction_Depth": [0.2,0.5,0.3], "Proximity_Nerve_p": 0.35, "Age_mu": 26},
         # Slightly more comfort with surgical, neutral on coronectomy/odontectomy
         "score_scale": {1:0.95, 2:1.05, 3:1.00, 4:1.00},
         "missingness": {"Tooth_Mobility": 0.05}},
    2:  {"name": "HighReferralClinic",
         "prevalence_shift": {"Impaction_Depth": [0.2,0.4,0.4], "Proximity_Nerve_p": 0.40, "Age_mu": 30},
         # Tends to prefer conservative nerve-sparing (coronectomy) and sectioning when indicated
         "score_scale": {1:0.90, 2:0.95, 3:1.10, 4:1.10},
         "missingness": {}},
    3:  {"name": "RuralGeneral",
         "prevalence_shift": {"Impaction_Depth": [0.4,0.45,0.15], "Proximity_Nerve_p": 0.25, "Age_mu": 27},
         # Slight tilt to simpler approaches locally
         "score_scale": {1:1.05, 2:1.00, 3:0.95, 4:0.95},
         "missingness": {"Bone_Density": 0.07}},
    4:  {"name": "OMFS_Experienced",
         "prevalence_shift": {"Impaction_Depth": [0.25,0.45,0.30], "Proximity_Nerve_p": 0.35, "Age_mu": 28},
         # Comfortable with complex surgery; coronectomy only when truly necessary
         "score_scale": {1:0.95, 2:1.05, 3:0.95, 4:1.05},
         "missingness": {}},
    5:  {"name": "OlderPopulation",
         "prevalence_shift": {"Age_mu": 35, "Proximity_Nerve_p": 0.32},
         "score_scale": {1:0.95, 2:1.00, 3:1.05, 4:1.05},
         "missingness": {}},
}

In [279]:
# write down meaning of numbers once
MULTI_TASK_CONFIG = {
    "tasks": {
        "surgical_decision": {
            "type": "classification",
            "classes": 4,
            "labels": ["Simple", "Surgical", "Coronectomy", "Odontectomy"]
        },
        "removal_indicated": {
            "type": "binary_classification", 
            "classes": 2,
            "labels": ["Monitor", "Remove"]
        },
        "risk_assessment": {
            "type": "regression",
            "targets": ["Risk_AlveolarOsteitis", "Risk_SecondaryInfection", 
                       "Risk_NerveDysesthesia", "Risk_Bleeding"]
        }
    }
}

In [280]:
def get_profile(client_id):
    return CLIENT_PROFILES.get(client_id,
        {"name": f"Clinic_{client_id}",
         "prevalence_shift": {},
         "score_scale": {1:1,2:1,3:1,4:1},
         "missingness": {}})

# global sizes
n_clients = 10
patients_per_client = 200

rows = []

# Generate per client

for c in range(1, n_clients+1):
    prof = get_profile(c)
    n = patients_per_client

    # Prevalence shifts / distributions
    Age_mu = prof["prevalence_shift"].get("Age_mu", 28)
    Proximity_p = prof["prevalence_shift"].get("Proximity_Nerve_p", 0.60)
    Depth_probs = prof["prevalence_shift"].get("Impaction_Depth", [0.45,0.45,0.1])

    # Features
    age = generate_age(n, mu=Age_mu)
    sex = generate_binary(n, 0.5)
    mandi_maxi= generate_binary(n, 0.48) # 0 mandi, 1 maxi

    pain = generate_binary(n, 0.5)
    swelling = generate_binary(n, 0.30)
    trismus = generate_binary(n, 0.20)
    pericoronitis = generate_binary(n, 0.75)

    caries_w = generate_binary(n, 0.20)
    caries_adj = generate_binary(n, 0.15)
    perio = generate_categorical(n, [1,2,3], [0.6,0.3,0.1])              # 1 healthy, 3 severe
    root_dev = generate_categorical(n, [1,2,3], [0.2,0.7,0.1])           # 1 incomplete, 2 complete, 3 complex/ 
    mobility = generate_categorical(n, [0,1,2], [0.7,0.2,0.1])           # 0 none, 2 severe

    # Angulation: 1 vertical, 2 mesioangular, 3 distoangular, 4 horizontal, 5 transverse (very rare)
    ang = generate_categorical(n, [1, 2, 3, 4, 5], [0.30, 0.40, 0.20, 0.09, 0.01])

    # Depth: 1 soft tissue, 2 partial bony, 3 deep bony
    depth = generate_categorical(n, [1, 2, 3], Depth_probs)

    # Gingival coverage subcategory for partial bony (depth == 2)
    # 0 = fully_covered, 1 = partially_covered (60%)
    gingival_cov = np.full(n, np.nan)  # default NaN for non-depth == 2
    mask_depth2 = (depth == 2)
    gingival_cov[mask_depth2] = generate_binary(mask_depth2.sum(), p=0.60)  # 60% partially covered

    prox_nerve = generate_binary(n, Proximity_p)                         # 1 = close/uncertain to IAN
    root_count = generate_categorical(n, [1,2,3,4], [0.1,0.5,0.3,0.1])
    root_curve = generate_binary(n, 0.70)                                # 1 = curved/divergent
    bone_density = generate_categorical(n, [1,2,3], [0.4,0.4,0.2])       # 3 = high

    cyst = generate_binary(n, 0.10)

    diabetes = generate_binary(n, 0.10)
    clotting = generate_binary(n, 0.02)
    smoking = generate_binary(n, 0.25)
    prev_issue = generate_binary(n, 0.10)

    # Osteo and Biophos
    # Age/sex-conditional osteoporosis
    def p_osteoporosis(a, s):
        # s: 0=male, 1=female
        if s == 0:  # male
            if a < 40: return 0.002
            if a < 50: return 0.010
            return 0.030
        else:       # female
            if a < 40: return 0.005
            if a < 50: return 0.020
            return 0.070

    osteoporosis = np.array([np.random.rand() < p_osteoporosis(age[i], sex[i]) for i in range(n)], dtype=int)

    # --- Bisphosphonates conditional on osteoporosis (+ mild age effect)
    def p_bisphosph(a, has_osteo):
        if has_osteo:
            base = 0.45
            # a bit higher uptake in older patients
            if a >= 50: base += 0.10
            return min(base, 0.70)  # cap
        else:
            return 0.003  # rare non-osteoporosis indications in this cohort

    bisphosph = np.array([np.random.rand() < p_bisphosph(age[i], osteoporosis[i]) for i in range(n)], dtype=int)

    for i in range(n):
        rows.append({
            "Client": c, "Patient": i+1,
            "Age": age[i], "Sex": sex[i], "Mandi_Maxi": mandi_maxi[i],
            "Pain": pain[i], "Swelling": swelling[i], "Trismus": trismus[i], "Pericoronitis": pericoronitis[i],
            "Caries_Wisdom": caries_w[i], "Caries_Adjacent": caries_adj[i],
            "Periodontal_Status": perio[i], "Root_Development": root_dev[i], "Tooth_Mobility": mobility[i],
            "Tooth_Angulation": ang[i], "Impaction_Depth": depth[i], "PartialBony_GingivalCoverage": (None if np.isnan(gingival_cov[i]) else int(gingival_cov[i])),
            "Proximity_Nerve": prox_nerve[i],
            "Root_Count": root_count[i], "Root_Curvature": root_curve[i], "Bone_Density": bone_density[i],
            "Cyst": cyst[i],
            "Diabetes": diabetes[i], "Osteoporosis": osteoporosis[i], "Clotting_Disorder": clotting[i],
            "Smoking": smoking[i], "Bisphosphonates": bisphosph[i],
            "Prev_Extraction_Issue": prev_issue[i],
        })

df = pd.DataFrame(rows)

# Optional: light missingness per profile
for c in range(1, n_clients+1):
    miss = get_profile(c)["missingness"]
    idx = df["Client"] == c
    for col, rate in miss.items():
        mask = (np.random.rand(idx.sum()) < rate)
        df.loc[idx, col] = df.loc[idx, col].mask(mask)

In [281]:
# Risk computation

import json
import os

def compute_risk_from_evidence(row, risk_type, risk_config, surgical_decision=None):
    """Compute risk based on evidence table configuration"""
    risk_data = risk_config[risk_type]
    risk = risk_data["base_incidence"]
    
    # Apply risk modifiers
    for feature, modifiers in risk_data["risk_modifiers"].items():
        if feature == "Age":
            # Handle age ranges
            age = row["Age"]
            for age_range in modifiers:
                if age_range["min"] <= age <= age_range["max"]:
                    risk *= age_range["mult"]
                    break
        else:
            # Handle categorical features and convert to string to match JSON keys
            feature_value = str(row[feature])
            if feature_value in modifiers:
                risk *= modifiers[feature_value]
    
    # Apply interactions
    for interaction in risk_data.get("interactions", []):
        conditions = interaction["conditions"]
        if all(str(row[feature]) in values for feature, values in conditions.items()):
            risk *= interaction["mult"]
    
    # Apply surgery modifiers if surgical decision is provided
    if surgical_decision is not None and "surgery_modifiers" in risk_data:
        surgery_key = str(surgical_decision)
        if surgery_key in risk_data["surgery_modifiers"]:
            risk *= risk_data["surgery_modifiers"][surgery_key]
    
    # Apply hard proximity gate for nerve risk
    if risk_type == "NerveDysesthesia" and row["Proximity_Nerve"] == 0:
        risk *= 0.30  # Clamp down when IAN not close

    # Apply hard gate for maxillary teeth - no nerve dysesthesia risk
    if risk_type == "NerveDysesthesia" and row["Mandi_Maxi"] == 1:
        risk = 0.0  # Maxillary teeth have zero nerve dysesthesia risk
        
    return min(1.0, max(0.0, risk))  # Clamp to [0,1]

In [282]:
def compute_removal_decision(row):
    """Binary decision: Should we remove this third molar at all? Uses config-based logic"""
    config = load_binary_config()
    
    # Start with base prevalence
    removal_prob = config["base_prevalence"]
    
    # Apply positive indications (increase probability)
    for feature, modifiers in config["positive_indications"].items():
        if feature == "Periodontal_Status":
            # Handle categorical features
            feature_value = str(row[feature])
            if feature_value in modifiers:
                removal_prob *= modifiers[feature_value]
        else:
            # Handle binary features
            if row[feature] == 1 and "1" in modifiers:
                removal_prob *= modifiers["1"]
    
    # Apply counter-indications (decrease probability)
    for feature, modifiers in config["counter_indications"].items():
        if feature == "Age":
            # Handle age ranges
            age = row["Age"]
            for age_range in modifiers:
                if age_range["min"] <= age <= age_range["max"]:
                    removal_prob *= age_range["mult"]
                    break
        elif feature == "Systemic_Risk":
            # Handle systemic risk factors - modifiers is a dict of risk factors
            for risk_factor, risk_modifiers in modifiers.items():
                if row[risk_factor] == 1 and "1" in risk_modifiers:
                    removal_prob *= risk_modifiers["1"]
        else:
            # Handle other features
            feature_value = str(row[feature])
            if feature_value in modifiers:
                removal_prob *= modifiers[feature_value]
    
    # Apply contextual modifiers
    for feature, modifiers in config["contextual_modifiers"].items():
        feature_value = str(row[feature])
        if feature_value in modifiers:
            removal_prob *= modifiers[feature_value]
    
    # Apply interactions
    for interaction in config.get("interactions", []):
        conditions = interaction["conditions"]
        match = True
        
        for feature, values in conditions.items():
            if feature == "Age":
                # Handle age range in interactions
                age = row["Age"]
                if "min" in values and "max" in values:
                    if not (values["min"] <= age <= values["max"]):
                        match = False
                        break
            else:
                # Handle other features
                feature_value = str(row[feature])
                if isinstance(values, list):
                    if feature_value not in values:
                        match = False
                        break
                else:
                    if feature_value != str(values):
                        match = False
                        break
        
        if match:
            removal_prob *= interaction["mult"]
    
    # Clamp probability to [0, 1]
    removal_prob = min(1.0, max(0.0, removal_prob))
    
    # Sample decision
    removal_decision = int(np.random.rand() < removal_prob)
    
    return removal_decision, removal_prob

In [283]:
# class semantics
# 1 = Simple extraction
# 2 = Surgical extraction (flap/bone)  -> has subtype
# 3 = Coronectomy (nerve-sparing, high IAN risk)
# 4 = Odontectomy (sectioning without high IAN risk)

def get_base_priors():
    """Get base priors from extraction config"""
    config = load_extraction_config()
    return {int(k): v for k, v in config["_base_priors"].items()}

def add(scores, cls, amt):
    scores[cls] += amt

def evaluate_rule_conditions(row, rule_data):
    """Evaluate rule conditions against patient data"""
    if "conditions" not in rule_data:
        return False
    
    conditions = rule_data["conditions"]
    for feature, values in conditions.items():
        if feature not in row:
            return False
        
        feature_value = row[feature]
        
        if isinstance(values, list) and len(values) > 0:
            if isinstance(values[0], str) and ">=" in values[0]:
                # Handle ">=3" type conditions
                threshold = int(values[0].replace(">=", ""))
                if feature_value < threshold:
                    return False
            elif feature_value not in values:
                return False
        else:
            if feature_value != values:
                return False
    
    if "additional_conditions" in rule_data:
        additional = rule_data["additional_conditions"]
        for feature, values in additional.items():
            if feature not in row:
                return False
            feature_value = row[feature]
            if isinstance(values[0], str) and ">=" in values[0]:
                threshold = int(values[0].replace(">=", ""))
                if feature_value < threshold:
                    return False
            elif feature_value not in values:
                return False
    
    return True

def apply_rule_category(row, scores, rule_category, config):
    """Apply a category of rules to the scores"""
    if rule_category not in config:
        return scores
    
    for rule_name, rule_data in config[rule_category].items():
        if rule_category == "age_rules" and rule_name == "Age_Scaling":
            # Special handling for age scaling rule
            age = row["Age"]
            age_factor = max(0.0, (age - 25) / 10.0)
            age_factor = min(age_factor, 2.0)  # Cap at 2.0 as per config
            for decision_class, effect in rule_data["effects"].items():
                add(scores, int(decision_class), effect * age_factor)
        elif evaluate_rule_conditions(row, rule_data):
            for decision_class, effect in rule_data["effects"].items():
                add(scores, int(decision_class), effect)
    return scores

def compute_scores_row(row):
    # Get base priors from config
    base_priors = get_base_priors()
    s = {1: base_priors[1], 2: base_priors[2], 3: base_priors[3], 4: base_priors[4]}
    
    config = load_extraction_config()
    
    # Apply each rule category
    s = apply_rule_category(row, s, "symptom_rules", config)
    s = apply_rule_category(row, s, "anatomy_rules", config)
    s = apply_rule_category(row, s, "pathology_rules", config)
    s = apply_rule_category(row, s, "mobility_rules", config)
    s = apply_rule_category(row, s, "root_development_rules", config)
    s = apply_rule_category(row, s, "periodontal_rules", config)
    s = apply_rule_category(row, s, "caries_rules", config)
    s = apply_rule_category(row, s, "systemic_rules", config)
    s = apply_rule_category(row, s, "age_rules", config)
    s = apply_rule_category(row, s, "ian_proximity_rules", config)
    s = apply_rule_category(row, s, "symptom_interactions", config)
    
    # Apply client preferences
    scales = get_profile(int(row["Client"]))["score_scale"]
    for k in s:
        s[k] *= scales.get(k, 1.0)
    
    return s

In [284]:
def decide_row(row, temperature=0.8, noise_sd=0.04):
    s = compute_scores_row(row)
    for k in s:
        s[k] += np.random.normal(0.0, noise_sd)
    probs = softmax([s[1], s[2], s[3], s[4]], temperature=temperature)
    decision = np.random.choice([1,2,3,4], p=probs)
    return decision, s, probs

In [285]:
risk_config = load_risk_config()

decisions, score1, score2, score3, score4, p1, p2, p3, p4, subtypes = [], [], [], [], [], [], [], [], [], []

removal_decisions, removal_probs = [], []
alveolar_risks, infection_risks, nerve_risks, bleeding_risks = [], [], [], []

for _, row in df.iterrows():
    # Primary surgical decision
    d, s, p = decide_row(row)
    decisions.append(d)
    score1.append(s[1]); score2.append(s[2]); score3.append(s[3]); score4.append(s[4])
    p1.append(p[0]); p2.append(p[1]); p3.append(p[2]); p4.append(p[3])
    
    # Surgical subtype
    if d == 2:
        depth = row["Impaction_Depth"]
        if depth == 1:
            subtypes.append("Soft_Tissue")
        elif depth == 2:
            subtypes.append("Partial_Bony")
        else:
            subtypes.append("Complete_Bony")
    else:
        subtypes.append(None)
    
    # Removal decision
    removal_decision, removal_prob = compute_removal_decision(row)
    removal_decisions.append(removal_decision)
    removal_probs.append(removal_prob)
    
    # Risk assessment
    alveolar_risk = compute_risk_from_evidence(row, "AlveolarOsteitis", risk_config, d)
    infection_risk = compute_risk_from_evidence(row, "SecondaryInfection", risk_config, d)
    nerve_risk = compute_risk_from_evidence(row, "NerveDysesthesia", risk_config, d)
    bleeding_risk = compute_risk_from_evidence(row, "Bleeding", risk_config, d)
    
    alveolar_risks.append(alveolar_risk)
    infection_risks.append(infection_risk)
    nerve_risks.append(nerve_risk)
    bleeding_risks.append(bleeding_risk)

# Add all columns to dataframe
# Existing columns
df["Surgical_Extraction_Type"] = decisions
df["Score_1"] = score1; df["Score_2"] = score2; df["Score_3"] = score3; df["Score_4"] = score4
df["Prob_1"] = p1; df["Prob_2"] = p2; df["Prob_3"] = p3; df["Prob_4"] = p4
df["Surg_2_Subtype"] = subtypes

df["Removal_Indicated"] = removal_decisions
df["Removal_Prob"] = removal_probs
df["Risk_AlveolarOsteitis"] = alveolar_risks
df["Risk_SecondaryInfection"] = infection_risks
df["Risk_NerveDysesthesia"] = nerve_risks
df["Risk_Bleeding"] = bleeding_risks

In [286]:
df.to_excel("../../data/raw/fed_recommenders_synthetic_dataset.xlsx", index=False)
df.to_csv("../../data/raw/fed_recommenders_synthetic_dataset.csv", index=False)

print("Saved: fed_recommenders_synthetic_dataset.xlsx and .csv to data/raw/")
print(f"Dataset shape: {df.shape}")

Saved: fed_recommenders_synthetic_dataset.xlsx and .csv to data/raw/
Dataset shape: (2000, 44)
